In [1]:
from pydantic_magic import pydantic_variant
from pydantic import BaseModel, Field
from typing import Literal

In [2]:
@pydantic_variant(annotations=Field(discriminator="type"))
class Test0(BaseModel):
    type: str
    test0: str = ""

class Test1(Test0):
    type: Literal["test1"] = "test1"
    test1: str = ""

class Test2(Test0):
    type: Literal["test2"] = "test2"
    test2: str = ""

class Test3(Test2):
    type: Literal["test3"] = "test3"
    test3: str = ""

In [3]:
t = Test0(type="test1")
print(t)
print(isinstance(t, Test0))
print(isinstance(t, Test1))
print(isinstance(t, Test2))
print(isinstance(t, Test3))

type='test1' test1=''
True
True
False
False


In [4]:

from typing import Literal
from pydantic import BaseModel, Field
from pydantic_magic import pydantic_variant

@pydantic_variant(annotations=Field(discriminator="name"))
class Fruit(BaseModel):
    name: str

class Apple(Fruit):
    name: Literal["apple"] = "apple"

class Orange(Fruit):
    name: Literal["orange"] = "orange"

class Cherry(Fruit):
    name: Literal["cherry"] = "cherry"


test = Fruit(name="apple")
assert isinstance(Fruit(name="apple"), Apple)
assert isinstance(Fruit(name="orange"), Orange)
assert isinstance(Fruit(name="cherry"), Cherry)
assert Apple().name == "apple"
assert Orange().name == "orange"
assert Cherry().name == "cherry"

In [1]:

from typing import Literal
from pydantic import BaseModel, Field
from pydantic_magic import pydantic_variant

# @pydantic_variant(annotations=Field(discriminator="name"))
@pydantic_variant
class Fruit(BaseModel):
    name: str
    thing_fruit: int = 0

class Apple(Fruit):
    name: Literal["apple"] = "apple"
    thing_apple: int = 1

class Orange(Fruit):
    name: Literal["orange"] = "orange"
    thing_orange: int = 2

class Cherry(Fruit):
    name: Literal["cherry"] = "cherry"
    thing_cherry: int = 3

class Berry(Fruit):
    pass
    thing_berry: int = 4

class Strawberry(Berry):
    name: Literal["strawberry"] = "strawberry"
    thing_strawberry: int = 5

print(f">>> {Fruit.model_variant}")

test = Fruit(name="apple")
assert isinstance(Fruit(name="apple"), Apple)
assert isinstance(Fruit(name="orange"), Orange)
assert isinstance(Fruit(name="cherry"), Cherry)
# assert isinstance(Fruit(name="berry"), Berry)
assert isinstance(Fruit(name="strawberry"), Strawberry)
assert Apple().name == "apple"
assert Orange().name == "orange"
assert Cherry().name == "cherry"
# assert Berry().name == "berry"
assert Strawberry().name == "strawberry"

::DEBUG:: cls=<class '__main__.Apple'>
>>> else
::DEBUG:: type(alternatives)=<class 'NoneType'>
::DEBUG:: alternatives=None
::DEBUG:: type(variant)=<class 'pydantic._internal._model_construction.ModelMetaclass'>
::DEBUG:: variant=<class '__main__.Apple'>
::DEBUG:: cls=<class '__main__.Orange'>
>>> else
::DEBUG:: type(alternatives)=<class 'pydantic._internal._model_construction.ModelMetaclass'>
::DEBUG:: alternatives=<class '__main__.Apple'>
::DEBUG:: type(variant)=<class 'typing._UnionGenericAlias'>
::DEBUG:: variant=typing.Union[__main__.Orange, __main__.Apple]
::DEBUG:: cls=<class '__main__.Cherry'>
>>> union
::DEBUG:: type(variant)=<class 'typing._UnionGenericAlias'>
::DEBUG:: variant=typing.Union[__main__.Cherry, __main__.Orange, __main__.Apple]
::DEBUG:: cls=<class '__main__.Berry'>
>>> union
::DEBUG:: type(variant)=<class 'typing._UnionGenericAlias'>
::DEBUG:: variant=typing.Union[__main__.Berry, __main__.Cherry, __main__.Orange, __main__.Apple]
::DEBUG:: cls=<class '__main__.Str

In [6]:

import math
from typing import Literal, Any
from abc import abstractmethod
from pydantic import BaseModel, model_validator, Field
from pydantic_magic import pydantic_variant

@pydantic_variant(annotations=Field(discriminator="type"))
# @pydantic_variant
class Shape(BaseModel):
    type: str

    @abstractmethod
    def area(self) -> float: ...

class Polygon(Shape):
    pass
    # type: Literal["polygon"] = "polygon"

    # def area(self) -> float:
    #     print("test area")


class Rectangle(Polygon):
    type: Literal["rectangle"] = "rectangle"

    length: float
    width: float

    def area(self) -> float:
        return self.length * self.width

class Square(Rectangle):
    type: Literal["square"] = "square"

    @model_validator(mode="before")
    def specify_one_length_width(cls, v: Any) -> Any:
        if not isinstance(v, dict):
            return v

        length = v.get("length")
        width = v.get("width")

        if length is not None:
            v["length"] = v["width"] = length
        elif width is not None:
            v["length"] = v["width"] = width
        else:
            raise ValueError("must specify one of 'length' or 'width'")

        return v

class Circle(Shape):
    type: Literal["circle"] = "circle"

    radius: float

    def area(self) -> float:
        return math.pi * self.radius**2


print(f">>> {Shape.model_variant}")

>>> typing.Annotated[typing.Union[__main__.Circle, __main__.Square, __main__.Rectangle], FieldInfo(annotation=NoneType, required=True, discriminator='type')]


In [7]:


# Shape()
#> ValidationError: Unable to extract tag using discriminator 'type'

# Shape(type="unknown")
#> ValidationError: Discriminator 'type' does not match any of the expected tags: 'circle', 'square', 'rectangle', 'polygon'

Shape(type="polygon")
#> TypeError: Can't instantiate abstract class Polygon without an implementation for abstract methods 'area', 'perimeter'

# Shape(type="rectangle")
#> ValidationError: Missing `rectangle.length` and `rectangle.width`

rectangle = Shape(type="rectangle", length=1.0, width=2.0)
assert type(rectangle) is Rectangle
assert isinstance(rectangle, Shape)
assert isinstance(rectangle, Polygon)
assert isinstance(rectangle, Rectangle)
assert not isinstance(rectangle, Square)
assert not isinstance(rectangle, Circle)
assert rectangle.area() == 2.0
assert rectangle.perimeter() == 6.0

square = Shape(type="square", length=1.0)
assert type(square) is Square
assert isinstance(square, Shape)
assert isinstance(square, Polygon)
assert isinstance(square, Rectangle)
assert isinstance(square, Square)
assert not isinstance(square, Circle)
assert square.area() == 1.0
assert square.perimeter() == 4.0

circle = Shape(type="circle", radius=1.0)
assert type(circle) is Circle
assert isinstance(circle, Shape)
assert not isinstance(circle, Polygon)
assert not isinstance(circle, Rectangle)
assert not isinstance(circle, Square)
assert isinstance(circle, Circle)
assert round(circle.area(), 2) == 3.14
assert round(circle.perimeter(), 2) == 6.28

ValidationError: 1 validation error for RootModel[Annotated[Union[Circle, Square, Rectangle], FieldInfo(annotation=NoneType, required=True, discriminator='type')]]
  Input tag 'polygon' found using 'type' does not match any of the expected tags: 'circle', 'square', 'rectangle' [type=union_tag_invalid, input_value={'type': 'polygon'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/union_tag_invalid

In [ ]:
class Test(type(None)):
    pass

In [ ]:
import math
from abc import abstractmethod
from typing import Any, Literal

from pydantic import BaseModel, Field, model_validator
from pydantic_magic import pydantic_variant

@pydantic_variant(annotations=Field(discriminator="type"))
class Shape(BaseModel):
    type: str

    @abstractmethod
    def area(self) -> float: ...

class Polygon(Shape):
    type: Literal["polygon"] = "polygon"
    sides: int

class Quadralateral(Polygon):
    type: Literal["quadralateral"] = "quadralateral"
    sides: Literal[4] = 4

class Rectangle(Quadralateral):
    type: Literal["rectangle"] = "rectangle"
    length: float
    width: float

    def area(self) -> float:
        return self.length * self.width

class Square(Rectangle):
    type: Literal["square"] = "square"

    @model_validator(mode="before")
    def specify_one_length_width(cls, v: Any) -> Any:
        if not isinstance(v, dict):
            return v

        length = v.get("length")
        width = v.get("width")

        if length is not None:
            v["length"] = v["width"] = length
        elif width is not None:
            v["length"] = v["width"] = width
        else:
            raise ValueError("must specify one of 'length' or 'width'")

        return v

class Circle(Shape):
    type: Literal["circle"] = "circle"
    radius: float

    def area(self) -> float:
        return math.pi * self.radius**2


# Shape()
#> ValidationError: Unable to extract tag using discriminator 'type'

# Shape(type="unknown")
#> ValidationError: Discriminator 'unknown' does not match any expected tags: 'circle', 'square', 'rectangle'

# Shape(type="polygon")
#> ValidationError: Discriminator 'polygon' does not match any expected tags: 'circle', 'square', 'rectangle'

# Shape(type="rectangle")
#> ValidationError: Missing `rectangle.length` and `rectangle.width`

rectangle = Shape(type="rectangle", length=1.0, width=2.0)
assert type(rectangle) is Rectangle
assert isinstance(rectangle, Shape)
assert isinstance(rectangle, Polygon)
assert isinstance(rectangle, Rectangle)
assert not isinstance(rectangle, Square)
assert not isinstance(rectangle, Circle)
assert rectangle.area() == 2.0

square = Shape(type="square", length=1.0)
assert type(square) is Square
assert isinstance(square, Shape)
assert isinstance(square, Polygon)
assert isinstance(square, Rectangle)
assert isinstance(square, Square)
assert not isinstance(square, Circle)
assert square.area() == 1.0

circle = Shape(type="circle", radius=1.0)
assert type(circle) is Circle
assert isinstance(circle, Shape)
assert not isinstance(circle, Polygon)
assert not isinstance(circle, Rectangle)
assert not isinstance(circle, Square)
assert isinstance(circle, Circle)
assert round(circle.area(), 2) == 3.14